In [15]:
from pathlib import Path
import os
import subprocess

REPOSITORY_URL = "https://github.com/Omid-Nejati/Locality-iN-Locality.git"
repository_candidates = [
    Path.cwd(),
    Path.cwd() / "Locality-iN-Locality-main",
    Path.cwd() / "Locality-iN-Locality",
]
repository_root = next(
    (path for path in repository_candidates if (path / "LNL_MoEx.py").exists()),
    None,
)

if repository_root is None:
    repository_root = Path.cwd() / "Locality-iN-Locality"
    subprocess.run(
        ["git", "clone", REPOSITORY_URL, str(repository_root)],
        check=True,
    )

os.chdir(repository_root)
print("Working directory:", Path.cwd())


Cloning into 'Locality-iN-Locality'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 36 (delta 8), reused 0 (delta 0), pack-reused 0
Unpacking objects: 100% (36/36), 40.40 KiB | 3.37 MiB/s, done.


In [ ]:
%pip install -q timm==0.9.16 einops scikit-learn


In [ ]:
import os
import math
import random
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF

from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, Subset
import timm
from timm.utils import ModelEmaV2

SEED = 42

def seed_everything(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True


In [ ]:
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("timm:", timm.__version__)
print("Device:", device)


## GTSRB

In [26]:
from urllib.request import urlretrieve

DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
GTSRB_URLS = {
    "GTSRB_Final_Training_Images.zip": (
        "https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/"
        "GTSRB_Final_Training_Images.zip"
    ),
    "GTSRB_Final_Test_Images.zip": (
        "https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/"
        "GTSRB_Final_Test_Images.zip"
    ),
    "GTSRB_Final_Test_GT.zip": (
        "https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/"
        "GTSRB_Final_Test_GT.zip"
    ),
}

for filename, url in GTSRB_URLS.items():
    destination = DATA_DIR / filename
    if not destination.exists():
        print("Downloading", filename)
        urlretrieve(url, destination)
    else:
        print("Already downloaded", filename)


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  263M  100  263M    0     0  11.6M      0  0:00:22  0:00:22 --:--:-- 13.2M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 84.8M  100 84.8M    0     0  9287k      0  0:00:09  0:00:09 --:--:-- 12.5M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 99620  100 99620    0     0  46726      0  0:00:02  0:00:02 --:--:-- 46726


In [27]:
import zipfile

dataset_extracted = (DATA_DIR / "GTSRB").exists() and (DATA_DIR / "GT-final_test.csv").exists()
if not dataset_extracted:
    for filename in GTSRB_URLS:
        with zipfile.ZipFile(DATA_DIR / filename) as archive:
            archive.extractall(DATA_DIR)
    print("GTSRB archives extracted.")
else:
    print("GTSRB dataset already extracted.")


Archive:  data/GTSRB_Final_Test_GT.zip
  inflating: data/GT-final_test.csv  


In [29]:
import csv

gtsrb_dir = DATA_DIR / "GTSRB"
test_images_dir = gtsrb_dir / "Final_Test" / "Images"
test_dir = gtsrb_dir / "test"
test_dir.mkdir(parents=True, exist_ok=True)

# ImageFolder needs the official test images arranged by class directory.
with (DATA_DIR / "GT-final_test.csv").open(newline="") as csv_file:
    for row in csv.DictReader(csv_file, delimiter=";"):
        class_dir = test_dir / f"{int(row['ClassId']):04d}"
        class_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(test_images_dir / row["Filename"], class_dir)


In [ ]:
class ResizeWithPadding:
    """Resize while preserving aspect ratio, then pad to a square image."""

    def __init__(self, size: int = 224, fill: int = 128):
        self.size = size
        self.fill = fill

    def __call__(self, image: Image.Image) -> Image.Image:
        width, height = image.size
        scale = self.size / max(width, height)

        new_width = max(1, round(width * scale))
        new_height = max(1, round(height * scale))

        image = TF.resize(
            image,
            [new_height, new_width],
            interpolation=transforms.InterpolationMode.BICUBIC,
            antialias=True,
        )

        horizontal_padding = self.size - new_width
        vertical_padding = self.size - new_height

        left = horizontal_padding // 2
        right = horizontal_padding - left
        top = vertical_padding // 2
        bottom = vertical_padding - top

        return TF.pad(
            image,
            [left, top, right, bottom],
            fill=self.fill,
            padding_mode="constant",
        )


NORMALIZE_MEAN = (0.5, 0.5, 0.5)
NORMALIZE_STD = (0.5, 0.5, 0.5)
RANDAUG_NUM_OPS = 2
RANDAUG_MAGNITUDE = 7

train_transform = transforms.Compose([
    ResizeWithPadding(size=224, fill=128),
    # Apply two moderate automatic policies per training image. Keeping this
    # before ToTensor lets RandAugment operate on the PIL image.
    transforms.RandAugment(
        num_ops=RANDAUG_NUM_OPS,
        magnitude=RANDAUG_MAGNITUDE,
        interpolation=transforms.InterpolationMode.BILINEAR,
        fill=128,
    ),
    transforms.ToTensor(),
    transforms.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
])

eval_transform = transforms.Compose([
    ResizeWithPadding(size=224, fill=128),
    transforms.ToTensor(),
    transforms.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
])


In [ ]:
TRAIN_ROOT = DATA_DIR / "GTSRB" / "Final_Training" / "Images"
TEST_ROOT = DATA_DIR / "GTSRB" / "test"

MICRO_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 64
NUM_WORKERS = 2

index_dataset = torchvision.datasets.ImageFolder(TRAIN_ROOT)
all_indices = np.arange(len(index_dataset))
all_targets = np.asarray(index_dataset.targets)

# GTSRB filenames have the form <TrackID>_<FrameID>.ppm. Track IDs are
# only unique inside a class, so include the class directory in the group key.
def gtsrb_track_group(sample_path: str) -> str:
    path = Path(sample_path)
    filename_parts = path.stem.split("_")
    if len(filename_parts) != 2 or not all(part.isdigit() for part in filename_parts):
        raise ValueError(
            f"Unexpected GTSRB training filename {path.name!r}; "
            "expected <TrackID>_<FrameID>.ppm."
        )
    return f"{path.parent.name}:{filename_parts[0]}"


all_groups = np.asarray([
    gtsrb_track_group(sample_path)
    for sample_path, _ in index_dataset.samples
])

TARGET_VALIDATION_SIZE = 4000
desired_n_splits = max(2, round(len(index_dataset) / TARGET_VALIDATION_SIZE))
groups_per_class = [
    len(np.unique(all_groups[all_targets == class_index]))
    for class_index in range(len(index_dataset.classes))
]
# Every class should be representable in every fold. This cap avoids asking
# for more folds than the least represented class has distinct tracks.
n_splits = min(desired_n_splits, min(groups_per_class))
if n_splits < 2:
    raise ValueError("At least two distinct tracks per class are required.")

splitter = StratifiedGroupKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=SEED,
)
candidate_splits = list(
    splitter.split(all_indices, all_targets, groups=all_groups)
)
# Group constraints make an exact 4,000-image holdout unlikely. Select the
# stratified fold whose size is closest to the previous validation target.
train_indices, validation_indices = min(
    candidate_splits,
    key=lambda split: abs(len(split[1]) - TARGET_VALIDATION_SIZE),
)

train_groups = set(all_groups[train_indices])
validation_groups = set(all_groups[validation_indices])
overlapping_groups = train_groups.intersection(validation_groups)
assert not overlapping_groups, (
    f"Track leakage detected between training and validation: "
    f"{sorted(overlapping_groups)[:5]}"
)
assert len(train_indices) + len(validation_indices) == len(index_dataset)

training_dataset = torchvision.datasets.ImageFolder(
    TRAIN_ROOT,
    transform=train_transform,
)
validation_dataset = torchvision.datasets.ImageFolder(
    TRAIN_ROOT,
    transform=eval_transform,
)
testset = torchvision.datasets.ImageFolder(
    TEST_ROOT,
    transform=eval_transform,
)

trainset = Subset(training_dataset, train_indices)
validationset = Subset(validation_dataset, validation_indices)

def make_loader(dataset, batch_size, *, shuffle=False, drop_last=False):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=device.type == "cuda",
        persistent_workers=NUM_WORKERS > 0,
        drop_last=drop_last,
    )

train_loader = make_loader(
    trainset, MICRO_BATCH_SIZE, shuffle=True, drop_last=True
)
validation_loader = make_loader(validationset, EVAL_BATCH_SIZE)
test_loader = make_loader(testset, EVAL_BATCH_SIZE)

NUM_CLASSES = len(index_dataset.classes)

print("Training images:", len(trainset))
print("Validation images:", len(validationset))
print("Stratified group folds:", n_splits)
print("Training tracks:", len(train_groups))
print("Validation tracks:", len(validation_groups))
print("Overlapping train/validation tracks:", len(overlapping_groups))
print("Test images:", len(testset))
print("Classes:", NUM_CLASSES)


## Fresh v3 + EMA: LNL-MoEx-RandAug-Ti for GTSRB Top-1 accuracy


## Train a new EMA checkpoint from scratch


In [ ]:
from LNL_MoEx import LNL_MoEx_Ti

# Train LNL-Ti from scratch; no external pretrained weights are loaded.
model = LNL_MoEx_Ti(
    pretrained=False,
    num_classes=NUM_CLASSES,
    drop_path_rate=0.10,
).to(device)


In [ ]:
@torch.inference_mode()
def calculate_top1(model: nn.Module, loader: DataLoader) -> float:
    """Evaluate clean Top-1 accuracy without retaining gradients."""
    model.eval()

    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
            logits = model(images)

        predictions = logits.argmax(dim=1)
        correct += predictions.eq(labels).sum().item()
        total += labels.size(0)

    return 100.0 * correct / total


In [ ]:
NUM_EPOCHS = 60
MIN_EPOCHS = 35
# Do not stop before cosine decay has entered its low-LR phase.
EARLY_STOP_PATIENCE = 15
EARLY_STOP_MIN_DELTA = 0.01  # percentage points
TARGET_VALIDATION_TOP1 = 99.5
TARGET_STABLE_EPOCHS = 2
WARMUP_EPOCHS = 5
ACCUMULATION_STEPS = 4

LEARNING_RATE = 3e-4
MIN_LR_FACTOR = 0.01
WEIGHT_DECAY = 0.05

MOEX_LAMBDA = 0.90
MOEX_PROBABILITY = 0.35
EMA_DECAY = 0.9998

# Prefer an explicit directory; otherwise use persistent Colab Drive when mounted.
# Use a local relative directory everywhere else.
checkpoint_dir_override = os.environ.get("CHECKPOINT_DIR")
if checkpoint_dir_override:
    CHECKPOINT_DIR = Path(checkpoint_dir_override).expanduser()
else:
    colab_drive_dir = Path("/content/drive/MyDrive")
    CHECKPOINT_DIR = (
        colab_drive_dir / "LNL_MoEx_checkpoints"
        if colab_drive_dir.exists()
        else Path.cwd() / "checkpoints"
    )
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / (
    f"LNL_MoEx_RandAug_Ti_GTSRB_fresh_v3_ema_seed{SEED}_top1.pth"
)
RESUME_TRAINING = (
    os.environ.get("RESUME_TRAINING", "0").strip().lower()
    not in {"0", "false", "no", "off"}
)
print("Training mode:", "resume" if RESUME_TRAINING else "fresh start")
print("Checkpoint:", CHECKPOINT_PATH)


In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

parameter_groups = {}

for name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue

    learning_rate = LEARNING_RATE

    no_weight_decay = (
        parameter.ndim == 1
        or name.endswith(".bias")
        or name in {"cls_token", "patch_pos", "pixel_pos"}
    )
    weight_decay = 0.0 if no_weight_decay else WEIGHT_DECAY

    group_key = (learning_rate, weight_decay)
    parameter_groups.setdefault(group_key, []).append(parameter)

optimizer = optim.AdamW(
    [
        {
            "params": parameters,
            "lr": learning_rate,
            "weight_decay": weight_decay,
        }
        for (learning_rate, weight_decay), parameters
        in parameter_groups.items()
    ],
    betas=(0.9, 0.999),
)


def learning_rate_factor(epoch: int) -> float:
    """Linear warmup followed by cosine decay."""
    if epoch < WARMUP_EPOCHS:
        return float(epoch + 1) / float(WARMUP_EPOCHS)

    progress = (
        epoch - WARMUP_EPOCHS
    ) / max(1, NUM_EPOCHS - WARMUP_EPOCHS)

    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return MIN_LR_FACTOR + (1.0 - MIN_LR_FACTOR) * cosine

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=learning_rate_factor,
)

scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")
ema_model = ModelEmaV2(model, decay=EMA_DECAY)


In [ ]:
def _cpu_state_dict(module: nn.Module) -> dict:
    """Copy model weights to CPU before storing them in a checkpoint."""
    return {name: value.detach().cpu().clone() for name, value in module.state_dict().items()}


def _atomic_torch_save(payload, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary_path)
    os.replace(temporary_path, path)


def _load_torch_checkpoint(path: Path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:  # Compatibility with older PyTorch versions.
        return torch.load(path, map_location=device)


def save_training_checkpoint(
    next_epoch: int,
    best_validation_top1: float,
    best_ema_model_state_dict: dict,
    no_improvement_epochs: int,
    target_streak: int,
) -> None:
    if best_ema_model_state_dict is None:
        best_ema_model_state_dict = _cpu_state_dict(ema_model.module)

    checkpoint = {
        "format_version": 4,
        "epoch": next_epoch,
        "best_validation_top1": best_validation_top1,
        "model_state_dict": model.state_dict(),
        "ema_model_state_dict": ema_model.module.state_dict(),
        "best_ema_model_state_dict": best_ema_model_state_dict,
        "ema_decay": EMA_DECAY,
        "optimizer_state_dict": optimizer.state_dict(),
        "no_improvement_epochs": no_improvement_epochs,
        "target_streak": target_streak,
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "rng_state": torch.get_rng_state(),
        "numpy_rng_state": np.random.get_state(),
        "python_rng_state": random.getstate(),
    }

    if torch.cuda.is_available():
        checkpoint["cuda_rng_state_all"] = torch.cuda.get_rng_state_all()

    _atomic_torch_save(checkpoint, CHECKPOINT_PATH)


def load_training_checkpoint():
    if not RESUME_TRAINING or not CHECKPOINT_PATH.exists():
        return 0, -1.0, None, 0, 0

    # The full checkpoint also stores Python/NumPy RNG state.
    checkpoint = _load_torch_checkpoint(CHECKPOINT_PATH)

    # Backward compatibility: older versions saved only a model state_dict.
    if "model_state_dict" not in checkpoint:
        model.load_state_dict(checkpoint, strict=True)
        ema_model.module.load_state_dict(checkpoint, strict=True)
        print(
            f"Loaded legacy weights-only checkpoint from {CHECKPOINT_PATH}; "
            "starting a new optimizer schedule."
        )
        return 0, -1.0, _cpu_state_dict(model), 0, 0

    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    ema_state_dict = checkpoint.get(
        "ema_model_state_dict",
        checkpoint["model_state_dict"],
    )
    ema_model.module.load_state_dict(ema_state_dict, strict=True)
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    scaler.load_state_dict(checkpoint.get("scaler_state_dict", {}))

    # Ensure optimizer state tensors follow the model device after loading.
    for state in optimizer.state.values():
        for name, value in state.items():
            if torch.is_tensor(value):
                state[name] = value.to(device)

    if "rng_state" in checkpoint:
        torch.set_rng_state(checkpoint["rng_state"].cpu())
    if "numpy_rng_state" in checkpoint:
        np.random.set_state(checkpoint["numpy_rng_state"])
    if "python_rng_state" in checkpoint:
        random.setstate(checkpoint["python_rng_state"])
    if torch.cuda.is_available() and "cuda_rng_state_all" in checkpoint:
        # map_location=device also moves saved RNG states to CUDA, but this
        # API requires CPU ByteTensors. Move them back before restoring.
        cuda_rng_states = [
            state.detach().cpu()
            for state in checkpoint["cuda_rng_state_all"]
        ]
        torch.cuda.set_rng_state_all(cuda_rng_states)

    best_ema_model_state_dict = checkpoint.get("best_ema_model_state_dict")
    if best_ema_model_state_dict is None:
        best_ema_model_state_dict = _cpu_state_dict(ema_model.module)

    start_epoch = int(checkpoint.get("epoch", 0))
    best_validation_top1 = float(checkpoint.get("best_validation_top1", -1.0))
    no_improvement_epochs = int(checkpoint.get("no_improvement_epochs", 0))
    target_streak = int(checkpoint.get("target_streak", 0))
    print(
        f"Resumed training from {CHECKPOINT_PATH} at epoch {start_epoch}/{NUM_EPOCHS}; "
        f"best validation Top-1: {best_validation_top1:.3f}%"
    )
    return (
        start_epoch,
        best_validation_top1,
        best_ema_model_state_dict,
        no_improvement_epochs,
        target_streak,
    )




In [ ]:
def compute_training_loss(images, labels):
    """Compute one raw-model loss, optionally using MoEx."""
    use_moex = torch.rand(1).item() < MOEX_PROBABILITY

    with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
        if use_moex:
            swap_index = torch.randperm(images.size(0), device=images.device)
            logits = model(
                images,
                swap_index=swap_index,
                moex_norm="pono",
                moex_epsilon=1e-5,
                moex_layer="stem",
                moex_positive_only=False,
            )
            swapped_labels = labels[swap_index]
            loss = (
                MOEX_LAMBDA * criterion(logits, labels)
                + (1.0 - MOEX_LAMBDA) * criterion(logits, swapped_labels)
            )
        else:
            loss = criterion(model(images), labels)

    return loss / ACCUMULATION_STEPS


def train_one_epoch():
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    total_steps = len(train_loader)

    for step, (images, labels) in enumerate(train_loader, start=1):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        loss = compute_training_loss(images, labels)
        running_loss += loss.item() * ACCUMULATION_STEPS

        scaler.scale(loss).backward()
        should_update = (
            step % ACCUMULATION_STEPS == 0
            or step == total_steps
        )
        if not should_update:
            continue

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        previous_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        # GradScaler lowers its scale when it skips an invalid optimizer step.
        # Update EMA only after a real parameter update.
        if scaler.get_scale() >= previous_scale:
            ema_model.update(model)
        optimizer.zero_grad(set_to_none=True)

    return running_loss / total_steps


(
    start_epoch,
    best_validation_top1,
    best_ema_model_state_dict,
    no_improvement_epochs,
    target_streak,
) = load_training_checkpoint()

for epoch in range(start_epoch, NUM_EPOCHS):
    training_loss = train_one_epoch()

    # EMA smooths late-training weight oscillations and is the model selected
    # for clean validation and final test.
    validation_top1 = calculate_top1(ema_model.module, validation_loader)
    previous_best = best_validation_top1

    if validation_top1 > best_validation_top1:
        best_validation_top1 = validation_top1
        best_ema_model_state_dict = _cpu_state_dict(ema_model.module)

    if validation_top1 > previous_best + EARLY_STOP_MIN_DELTA:
        no_improvement_epochs = 0
    else:
        no_improvement_epochs += 1

    target_streak = target_streak + 1 if validation_top1 > TARGET_VALIDATION_TOP1 else 0
    scheduler.step()

    # Save the latest state every epoch so an interrupted run can resume safely.
    save_training_checkpoint(
        next_epoch=epoch + 1,
        best_validation_top1=best_validation_top1,
        best_ema_model_state_dict=best_ema_model_state_dict,
        no_improvement_epochs=no_improvement_epochs,
        target_streak=target_streak,
    )

    print(
        f"Epoch {epoch + 1:03d}/{NUM_EPOCHS} | "
        f"train loss: {training_loss:.4f} | "
        f"EMA validation Top-1: {validation_top1:.3f}% | "
        f"best: {best_validation_top1:.3f}% | "
        f"plateau: {no_improvement_epochs}/{EARLY_STOP_PATIENCE}"
    )

    reached_target = (
        epoch + 1 >= MIN_EPOCHS
        and target_streak >= TARGET_STABLE_EPOCHS
    )
    reached_plateau = (
        epoch + 1 >= MIN_EPOCHS
        and no_improvement_epochs >= EARLY_STOP_PATIENCE
    )

    if reached_target or reached_plateau:
        reason = (
            f"target > {TARGET_VALIDATION_TOP1:.2f}% was stable"
            if reached_target
            else f"no meaningful improvement for {EARLY_STOP_PATIENCE} epochs"
        )
        print(f"Early stopping: {reason}.")
        break

print(f"Best validation Top-1: {best_validation_top1:.3f}%")


## Final clean Top-1 test and plug-and-play checkpoint verification


In [ ]:
verification_model = LNL_MoEx_Ti(
    pretrained=False,
    num_classes=NUM_CLASSES,
).to(device)

checkpoint = _load_torch_checkpoint(CHECKPOINT_PATH)
state_dict = checkpoint.get("best_ema_model_state_dict")
if state_dict is None:
    state_dict = checkpoint.get(
        "ema_model_state_dict",
        checkpoint.get("model_state_dict", checkpoint),
    )
verification_model.load_state_dict(state_dict, strict=True)

test_top1 = calculate_top1(
    verification_model,
    test_loader,
)

print(f"Top-1 accuracy: {test_top1:.3f}%")
if test_top1 > TARGET_VALIDATION_TOP1:
    print(f"Target > {TARGET_VALIDATION_TOP1:.1f}%: achieved.")
else:
    print(f"Target > {TARGET_VALIDATION_TOP1:.1f}%: not achieved yet.")


In [ ]:
try:
    from google.colab import files
    files.download(str(CHECKPOINT_PATH))
except ImportError:
    print(f"Checkpoint saved at: {CHECKPOINT_PATH.resolve()}")
